In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))


In [2]:
exports_folder = Path("dp_export_tests")
areas_export_path = exports_folder / "00_add_areas"
clean_db_export_path: Path = exports_folder / "01_cleanup_db"

In [3]:
df = pl.scan_parquet(areas_export_path)

df.filter(pl.col("i") == 2).collect()

lod_level,lod_code,face,i,j,row_id,uint8_reflectance,32bit_reflectance,positions_x,positions_y,positions_z,area
u8,str,str,u32,u32,u32,u8,f32,f32,f32,f32,f32
1,"""A""","""negx""",2,625,13978,108,0.018777,0.121296,0.143183,-0.143118,0.00074
1,"""A""","""negx""",2,645,13978,166,0.02885,0.120656,0.143253,-0.143188,0.000731
1,"""A""","""negx""",2,647,13978,167,0.02895,0.120593,0.143257,-0.143193,0.000823
1,"""A""","""negx""",2,648,13978,163,0.028212,0.12056,0.143263,-0.143193,0.000889
1,"""A""","""negx""",2,610,13978,147,0.025441,0.121759,0.143115,-0.143045,0.000851
…,…,…,…,…,…,…,…,…,…,…,…
3,"""BBB""","""posz""",2,8062,1282073,210,0.037474,-0.136194,-0.131916,-0.136151,0.000706
4,"""BBBB""","""posz""",2,8121,1297706,141,0.024421,-0.135762,-0.133455,-0.135724,0.000646
4,"""BBBB""","""posz""",2,8123,1297706,133,0.023053,-0.135752,-0.133507,-0.135705,0.000665


In [4]:
from boulder_statistics.analysis.data_product_encyclopedia import FACES

df = df.with_columns(
    pl.col("face").cast(pl.Enum(FACES))
).select(
    pl.col("face"),
    pl.col("i"),
    pl.col("j"),
    pl.col("row_id"),
    pl.col("uint8_reflectance").alias("u8_R"),
    pl.col("32bit_reflectance").alias("32f_R"),
    pl.col("positions_x").alias("x"),
    pl.col("positions_y").alias("y"),
    pl.col("positions_z").alias("z"),
    pl.col("area")
).with_columns(
    [
        (pl.arctan2(pl.col("y"), pl.col("x")) * 180 / np.pi).alias("long"),
        (
            pl.arctan2(
                pl.col("z"),
                (pl.col("x") ** 2 + pl.col("y") ** 2).sqrt()
            ) * 180 / np.pi
        ).alias("lat"),
    ]
)

df.sink_parquet(clean_db_export_path, engine = "streaming")

In [5]:
pl.scan_parquet(clean_db_export_path).filter(pl.col("i") == 2).collect()

face,i,j,row_id,u8_R,32f_R,x,y,z,area,long,lat
enum,u32,u32,u32,u8,f32,f32,f32,f32,f32,f32,f32
"""negx""",2,625,13978,108,0.018777,0.121296,0.143183,-0.143118,0.00074,49.730808,-37.331738
"""negx""",2,645,13978,166,0.02885,0.120656,0.143253,-0.143188,0.000731,49.893871,-37.398129
"""negx""",2,647,13978,167,0.02895,0.120593,0.143257,-0.143193,0.000823,49.909416,-37.404655
"""negx""",2,648,13978,163,0.028212,0.12056,0.143263,-0.143193,0.000889,49.918354,-37.407139
"""negx""",2,610,13978,147,0.025441,0.121759,0.143115,-0.143045,0.000851,49.609577,-37.281113
…,…,…,…,…,…,…,…,…,…,…,…
"""posz""",2,8062,1282073,210,0.037474,-0.136194,-0.131916,-0.136151,0.000706,-135.913986,-35.681034
"""posz""",2,8121,1297706,141,0.024421,-0.135762,-0.133455,-0.135724,0.000646,-135.491028,-35.486649
"""posz""",2,8123,1297706,133,0.023053,-0.135752,-0.133507,-0.135705,0.000665,-135.477783,-35.478565
